> **Status: currently blocked on this board.** PYNQ 3.x needs `pyxrt` (XRT's Python bindings), which isn't available on this lab's KV260 image -- `pynq.Overlay(...)` below fails with `RuntimeError: No Devices Found` / `NameError: name 'pyxrt' is not defined`. The path that actually works today is `validate_cynq.cpp` (C++, via [CYNQ](https://github.com/ECASLab/cynq)) -- see `README.md`. This notebook is kept ready to run as-is if `pyxrt` ever becomes available.

# DFS accelerator -- on-board validation (issue #67)

Runs the same 21 golden cases the HLS cosim testbench checks
(`src/hls/tb/dfs_accel_tb.cpp`, issue #63), but against the real
accelerator on this KV260 instead of Vitis HLS's simulator.

**Before running:** this notebook, `dfs_system.bit`, `dfs_system.hwh`,
`cases.json` and `driver.py` must all sit in the same directory (see
`src/onboard/driver.py`'s header for why the register map here is *not*
the same as the SystemC/TLM one in `accelerator_driver.h`).

A case only counts as a **failure** if its result value doesn't match
the golden `expected_value`. A traversal-counter mismatch alone is
reported (status `ok*`) but doesn't fail the case -- same distinction
the cosim testbench makes, since counters aren't part of what #67
validates.

In [ ]:
import json

import pynq

from driver import DfsAccelDriver

## Load the overlay and the case fixture

`pynq.Overlay` auto-loads the `.hwh` with the same basename next to the
`.bit` -- see `build_bitstream.tcl`, which always exports them paired.

In [ ]:
overlay = pynq.Overlay("dfs_system.bit")
driver = DfsAccelDriver(overlay)

with open("cases.json") as f:
    cases = json.load(f)

print(f"loaded {len(cases)} cases")

## Run all 21 cases on the accelerator

In [ ]:
rows_out = []
failures = 0
counter_mismatches = 0

print(f"{'algorithm':<32} {'case':<18} {'expect':>8} {'hw':>8}  status")

for case in cases:
    result = driver.run_case(case)

    value_ok = result["value"] == case["expected_value"]
    counters_ok = (
        result["expanded_nodes"] == case["expected_expanded_nodes"]
        and result["visited_cells"] == case["expected_visited_cells"]
        and result["peak_stack_depth"] == case["expected_peak_stack_depth"]
    )

    if not value_ok:
        failures += 1
    elif not counters_ok:
        counter_mismatches += 1

    status = "FAIL" if not value_ok else ("ok" if counters_ok else "ok*")
    rows_out.append((case, result, status))

    print(
        f"{case['algorithm']:<32} {case['case']:<18} "
        f"{case['expected_value']:>8} {result['value']:>8}  {status}"
    )

## Summary

In [ ]:
total = len(cases)
print(f"{total - failures}/{total} cases match the golden result (on-board)")
if counter_mismatches:
    print(f"{counter_mismatches} case(s) marked ok* differ in traversal counters")
if failures:
    print(f"{failures} case(s) FAILED -- see the table above")